In [32]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict, Annotated
from pydantic import BaseModel, Field
from langchain_google_genai import ChatGoogleGenerativeAI
import os
from dotenv import load_dotenv
import operator

In [33]:
llm = ChatGoogleGenerativeAI(
    model="gemini-3.1-flash-lite",
    google_api_key=os.getenv("GOOGLE_GENAI_KEY"),
)

In [34]:
class EvaluationSchema(BaseModel):
    feedback: str = Field(description="Detailed feedback from the essay")
    score: int = Field(description="Score out of 10 for the essay", ge=0, le=10)


In [35]:
structuredModel = llm.with_structured_output(EvaluationSchema)

In [36]:
essay = """
Artificial Intelligence (AI) is transforming the modern world by improving healthcare, education, business, and transportation. It enables machines to perform tasks that normally require human intelligence, making processes faster, more accurate, and more efficient.

In healthcare, AI helps doctors diagnose diseases early and provide personalized treatments. In education, it offers personalized learning experiences and improves accessibility. Businesses use AI to automate repetitive tasks, enhance customer service, and make better decisions using data analysis.

Despite its many benefits, AI also presents challenges such as job displacement, data privacy concerns, and algorithmic bias. Therefore, governments and organizations must develop ethical guidelines and regulations to ensure AI is used responsibly.

In conclusion, AI has the potential to greatly improve society if developed and implemented ethically. By balancing innovation with responsibility, AI can contribute to a more efficient, inclusive, and sustainable future.
"""

In [37]:
prompt = f"Evaluate the following essay and provide feedback and a score out of 10:\n\n{essay}"

structuredModel.invoke(prompt).feedback

'The essay provides a clear and well-structured overview of the role of AI in modern society. It effectively balances the discussion between benefits and challenges. The writing is concise and professional. To improve, the author could provide more concrete examples or specific case studies to substantiate the claims, and the conclusion could offer a more nuanced reflection on how ethical frameworks might be implemented in practice.'

In [38]:
class UPSCState(TypedDict):
    essay: str
    language_feedback: str
    analysis_feedback: str
    clarity_feedback: str
    overall_feedback: int
    individual_scores: Annotated[list[int], operator.add]
    average_score: float

In [39]:
def evaluate_language(state: UPSCState):
    prompt = f"Evaluate the language of the following essay and provide feedback and a score out of 10:\n\n{state['essay']}"
    result = structuredModel.invoke(prompt)
    
    return {
        'language_feedback': result.feedback,
        'individual_scores': [result.score]
    }

def evaluate_analysis(state: UPSCState):
    prompt = f"Evaluate the depth of following essay and provide feedback and a score out of 10:\n\n{state['essay']}"
    result = structuredModel.invoke(prompt)
    
    return {
        'analysis_feedback': result.feedback,
        'individual_scores': [result.score]
    }
def evaluate_thought(state: UPSCState):
    prompt = f"Evaluate the clarity of thought of the following essay and provide feedback and a score out of 10:\n\n{state['essay']}"
    result = structuredModel.invoke(prompt)
    
    return {
        'clarity_feedback': result.feedback,
        'individual_scores': [result.score]
    }

def final_evaluation(state: UPSCState):

    #Summary feedback
    prompt = f"""Based on the following feedback create a summarized feedback \n
            Language_feedback : {state['language_feedback']}, \n
            Analysis_feedback : {state['analysis_feedback']}, \n
            Clarity Of Though feedback : {state['clarity_feedback']}    
    """

    result = llm.invoke(prompt).content

    #Avergae Score
    avg = sum(state['individual_scores'])/len(state['individual_scores'])

    return {
        'overall_feedback': result,
        'average_score': avg
    }

In [40]:
graph = StateGraph(UPSCState)

#Add Node
graph.add_node('evaluate_language', evaluate_language)
graph.add_node('evaluate_analysis', evaluate_analysis)
graph.add_node('evaluate_thought', evaluate_thought)
graph.add_node('final_evaluation',final_evaluation)
#dges
graph.add_edge(START, 'evaluate_language')
graph.add_edge(START, 'evaluate_analysis')
graph.add_edge(START, 'evaluate_thought')

graph.add_edge('evaluate_language', 'final_evaluation')
graph.add_edge('evaluate_analysis', 'final_evaluation')
graph.add_edge('evaluate_thought', 'final_evaluation')

worfklow = graph.compile()


In [41]:
initial_state = {
    'essay': essay
}

final_state = worfklow.invoke(initial_state)

final_state

{'essay': '\nArtificial Intelligence (AI) is transforming the modern world by improving healthcare, education, business, and transportation. It enables machines to perform tasks that normally require human intelligence, making processes faster, more accurate, and more efficient.\n\nIn healthcare, AI helps doctors diagnose diseases early and provide personalized treatments. In education, it offers personalized learning experiences and improves accessibility. Businesses use AI to automate repetitive tasks, enhance customer service, and make better decisions using data analysis.\n\nDespite its many benefits, AI also presents challenges such as job displacement, data privacy concerns, and algorithmic bias. Therefore, governments and organizations must develop ethical guidelines and regulations to ensure AI is used responsibly.\n\nIn conclusion, AI has the potential to greatly improve society if developed and implemented ethically. By balancing innovation with responsibility, AI can contrib